# Assignment 6 - Kafka Stream Processing for Yellow Taxi Trips

Sara Milovanova, Biljana Vitanova


In [26]:
import json
import time
from pathlib import Path

import pandas as pd
from confluent_kafka import Consumer
from confluent_kafka.admin import AdminClient, NewTopic

from config import *

### 1. Stream Data with Kafka Producer and Consumer

We use a subset of the dataset containing taxi trips from the first two days of September 2019. It contains 273,852 records in total. The producer sends one Kafka message per taxi trip. The messages are ordered by `tpep_pickup_datetime`. 

In [17]:
df = pd.read_parquet("september_first_two_days.parquet")
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 273852
Columns: 17


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,PU_Borough,PU_Zone,DO_Borough,DO_Zone,trip_distance,fare_amount,tip_amount,total_amount,pickup_time_hour,temperature_c,precipitation_mm,pickup_business_count,dropoff_business_count
0,2019-09-01,2019-09-01 00:17:16,229,112,Manhattan,Sutton Place/Turtle Bay North,Brooklyn,Greenpoint,4.19,16.0,1.50,21.30,2019-09-01,23.5,0.0,193,285
1,2019-09-01,2019-09-01 00:05:47,148,256,Manhattan,Lower East Side,Brooklyn,Williamsburg (South Side),2.00,8.0,2.00,13.80,2019-09-01,23.5,0.0,223,181
2,2019-09-01,2019-09-01 00:26:19,256,37,Brooklyn,Williamsburg (South Side),Brooklyn,Bushwick South,3.82,17.5,3.76,24.51,2019-09-01,23.5,0.0,181,323
3,2019-09-01,2019-09-01 00:05:26,231,261,Manhattan,TriBeCa/Civic Center,Manhattan,World Trade Center,1.16,6.0,0.00,9.80,2019-09-01,23.5,0.0,205,77
4,2019-09-01,2019-09-01 00:33:02,74,226,Manhattan,East Harlem North,Queens,Sunnyside,10.03,34.0,0.00,41.42,2019-09-01,23.5,0.0,288,357


In [4]:
df[[
    "tpep_pickup_datetime",
    "PULocationID",
    "DOLocationID",
    "PU_Borough",
    "PU_Zone",
    "trip_distance",
    "fare_amount",
    "tip_amount",
]].sort_values("tpep_pickup_datetime").head(10)

,tpep_pickup_datetime,PULocationID,DOLocationID,PU_Borough,PU_Zone,trip_distance,fare_amount,tip_amount
0,2019-09-01 00:00:00,229,112,Manhattan,Sutton Place/Turtle Bay North,4.19,16.0,1.50
1,2019-09-01 00:00:00,148,256,Manhattan,Lower East Side,2.00,8.0,2.00
2,2019-09-01 00:00:00,256,37,Brooklyn,Williamsburg (South Side),3.82,17.5,3.76
3,2019-09-01 00:00:00,231,261,Manhattan,TriBeCa/Civic Center,1.16,6.0,0.00
4,2019-09-01 00:00:00,74,226,Manhattan,East Harlem North,10.03,34.0,0.00
5,2019-09-01 00:00:00,114,80,Manhattan,Greenwich Village South,5.10,22.0,0.00
7,2019-09-01 00:00:01,132,95,Queens,JFK Airport,8.40,25.0,0.00
8,2019-09-01 00:00:01,79,148,Manhattan,East Village,0.40,4.5,1.65
6,2019-09-01 00:00:01,211,230,Manhattan,SoHo,2.90,14.5,3.65
9,2019-09-01 00:00:02,181,229,Brooklyn,Park Slope,7.98,25.0,5.76


#### 1.1 Kafka Topics

We use 6 topics: 
 - `yellow-taxi-events` - Raw taxi trip stream written by the producer, all processors consume from this topic.

- `taxi-borough-stats` - Quix Streams processor writes rolling statistics grouped by borough.

- `taxi-location-stats` - Quix Streams processor writes rolling statistics for the top 10 pickup/dropoff locations.

- `taxi-borough-stats-python` - Regular Python processor (without Quix) writes borough statistics.

- `taxi-location-stats-python` - Regular Python processor writes location statistics without Quix Streams.

- `taxi-clusters` - Online clustering processor writes cluster labels/assignments for taxi trips.

In [5]:
topics = {
    "raw taxi events": TOPIC_RAW,
    "Quix borough statistics": TOPIC_BOROUGH_STATS,
    "Quix top-location statistics": TOPIC_LOCATION_STATS,
    "regular Python borough statistics": TOPIC_PY_BOROUGH_STATS,
    "regular Python top-location statistics": TOPIC_PY_LOCATION_STATS,
    "stream clustering": TOPIC_CLUSTERS,
}

topics

{'raw taxi events': 'yellow-taxi-events',
 'Quix borough statistics': 'taxi-borough-stats',
 'Quix top-location statistics': 'taxi-location-stats',
 'regular Python borough statistics': 'taxi-borough-stats-python',
 'regular Python top-location statistics': 'taxi-location-stats-python',
 'stream clustering': 'taxi-clusters'}

#### Reset Topics

We use this helper to delete and recreate Kafka topics before each run, avoiding cached or old messages from previous executions.

In [6]:
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

ALL_TOPICS = list(topics.values())

print("Deleting topics...")
admin.delete_topics(ALL_TOPICS)
time.sleep(3)

print("Recreating topics...")
new_topics = [NewTopic(topic=t, num_partitions=1, replication_factor=1) for t in ALL_TOPICS]
admin.create_topics(new_topics)

print("Done")

Deleting topics...
Recreating topics...
Done


#### 1.2 Producer

`producer.py` reads the Parquet file, sorts trips by pickup timestamp, converts each row to JSON, and sends the messages to the raw Kafka topic `yellow-taxi-events`.

In [7]:
print(Path("producer.py").read_text())

"""
    The script acts as a producer, reading the taxi dataset, orders trips by pickup
    time, converts each row to JSON, and sends each trip to the raw Kafka topic.
"""

import json
import time
import pandas as pd
from confluent_kafka import Producer
from config import *


df = pd.read_parquet("september_first_two_days.parquet")
df = df.sort_values("tpep_pickup_datetime")


def row_to_json(row):
    """Convert a taxi trip row into a JSON string Kafka can send."""

    d = row.to_dict()

    for k, v in d.items():

        if hasattr(v, "isoformat"):
            d[k] = v.isoformat()

        elif isinstance(v, float) and pd.isna(v):
            d[k] = None

    return json.dumps(d)



producer = Producer({"bootstrap.servers": BOOTSTRAP})

print("Streaming taxi events...")

for i, (_, row) in enumerate(df.iterrows()):
    
  
    producer.produce(TOPIC_RAW, key=str(row["PULocationID"]), value=row_to_json(row))

    if i % 5000 == 0:
        producer.poll(0)

        print(f"Sent {i}/

#### 1.3 Basic Consumer

`basic_consumer.py` reads raw taxi events from Kafka and prints trip information, without using Faust. 

In [8]:
print(Path("basic_consumer.py").read_text())

"""
    This script acts as a consumer, reading raw NYC taxi trip events.
    It subscribes to the raw taxi events topic, continuously reads incoming messages,
    and prints trip information.
"""

import json
from confluent_kafka import Consumer
from config import *

consumer = Consumer(
    {
        "bootstrap.servers": BOOTSTRAP,
        "group.id": "basic-group",
        "auto.offset.reset": "earliest",
    }
)

consumer.subscribe([TOPIC_RAW])

print("Listening for taxi events...")

count = 0

try:

    while True:

        msg = consumer.poll(1.0)

        if msg is None:
            continue

        if msg.error():
            print(msg.error())
            continue

        event = json.loads(msg.value().decode("utf-8"))

        print(
            f"[{count}] "
            f"{event['PU_Borough']} | "
            f"fare={event['fare_amount']:.2f} | "
            f"dist={event['trip_distance']:.2f}"
        )

        count += 1

except KeyboardInterrupt:
    print("Stopping co

### 2. Quix Streams Processing

`quix_streams.py` consumes raw taxi trips and computes windowed descriptive statistics using 30-second non-overlapping time windows. For each group, it computes count, mean, standard deviation, minimum, and maximum for three attributes: `trip_distance`, `fare_amount`, and `tip_amount`. The processed statistics are written back to Kafka topics.

In [9]:
print(Path("quix_streams.py").read_text())

import math
import argparse

from datetime import timedelta

from quixstreams import Application

from config import *

FIELDS = ["trip_distance", "fare_amount", "tip_amount"]

borough_seen = 0
location_seen = 0
location_matched = 0


def initializer(value):
    """Create the first accumulator for each borough/location in each window."""

    acc = {"n": 0}
    for f in FIELDS:

        acc[f"sum_{f}"] = 0.0
        acc[f"ssq_{f}"] = 0.0
        acc[f"min_{f}"] = float("inf")
        acc[f"max_{f}"] = float("-inf")

    return reducer(acc, value)


def reducer(acc, value):
    """Update the accumulator with one incoming taxi trip."""

    acc["n"] += 1
    for f in FIELDS:

        v = value.get(f) or 0.0

        acc[f"sum_{f}"] += v
        acc[f"ssq_{f}"] += v**2
        acc[f"min_{f}"] = min(acc[f"min_{f}"], v)
        acc[f"max_{f}"] = max(acc[f"max_{f}"], v)

    return acc


def finalize(window_result):

    """Calculate mean, standard deviation, min, and max for a completed win

#### 2.1 Borough Streaming

The borough-level stream groups taxi trips by pickup borough (`PU_Borough`). Each borough (Manhatta, Brooklyn, Queens) is processed as a separate group.

In [10]:
borough_counts = (
    df["PU_Borough"]
    .value_counts()
    .reset_index()
)

borough_counts.columns = ["PU_Borough", "trip_count"]
borough_counts

,PU_Borough,trip_count
0,Manhattan,241037
1,Queens,29110
2,Brooklyn,3705


#### 2.2 Top 10 Locations Streaming

The selected top 10 locations are based on the most frequent pickup/dropoff activity in the stream dataset.

In [11]:
top_location_info = (
    df[["PULocationID", "PU_Borough", "PU_Zone"]]
    .drop_duplicates()
    .set_index("PULocationID")
    .loc[TOP10_LOCATIONS]
)

top_location_info

,PU_Borough,PU_Zone
PULocationID,,
186,Manhattan,Penn Station/Madison Sq West
48,Manhattan,Clinton East
132,Queens,JFK Airport
230,Manhattan,Times Sq/Theatre District
161,Manhattan,Midtown Center
170,Manhattan,Murray Hill
79,Manhattan,East Village
68,Manhattan,East Chelsea
236,Manhattan,Upper East Side North


#### 2.3 Stream Clustering

We implement a simple online k-means algorithm `stream_clustering.py`. Each taxi trip is converted into a feature vector using **distance**, **fare**, **tip**, **total amount**, **pickup business count**, and **dropoff business count**.  The model updates as each new trip arrives and writes the cluster assignment to Kafka.

Each feature is scaled before clustering:

$$
x'_j = \frac{x_j}{s_j}
$$

where $x_j$ is the original feature value and $s_j$ is the manually selected scale factor for that feature. The scale factors were chosen based on the typical value ranges of the features. This prevents features with larger numeric values from dominating the distance calculation.

In [12]:
CLUSTER_FEATURES = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "pickup_business_count",
    "dropoff_business_count",
]

feature_stats = df[CLUSTER_FEATURES].agg(["mean", "std", "min", "max"]).T
feature_stats

,mean,std,min,max
trip_distance,3.452271,4.300726,0.0,79.50
fare_amount,13.460691,12.122381,0.0,423.00
tip_amount,2.041913,2.819658,0.0,280.00
total_amount,19.139523,14.821147,0.0,837.93
pickup_business_count,176.660094,109.805671,0.0,693.00
dropoff_business_count,188.125341,110.544152,0.0,693.00


In [13]:
print(Path("stream_clustering.py").read_text())

"""
    Consume taxi trip events from Kafka, apply online k-means clustering, and write
    the assigned cluster for each trip back to a Kafka topic.
"""

import json
import math
import time

from confluent_kafka import Consumer, Producer

from config import *


FEATURE_SCALES = {
    "trip_distance": 12.0,
    "fare_amount": 40.0,
    "tip_amount": 8.0,
    "total_amount": 50.0,
    "pickup_business_count": 400.0,
    "dropoff_business_count": 400.0,
}


class OnlineKMeans:
    def __init__(self, n_clusters):
        self.n_clusters = n_clusters
        self.centers = []
        self.counts = []

    def update(self, x):
        """Assign one feature vector to the nearest cluster and update its center."""
        if len(self.centers) < self.n_clusters:
            cluster_id = len(self.centers)
            self.centers.append(list(x))
            self.counts.append(1)
            return cluster_id, 0.0

        distances = [self._distance(x, center) for center in self.centers]
       

### 3. Standard Python Stream Processing

`regular_python_stats.py` uses only regular Python and `confluent-kafka` to consume raw events, maintain time-window accumulators, calculate descriptive statistics, and write the results back to Kafka.

In [ ]:
print(Path("regular_python_stats.py").read_text())

### 4. Read Processed Results

In [27]:
def consume_topic(topic, limit=20, seconds=30):
    consumer = Consumer({
        "bootstrap.servers": BOOTSTRAP,
        "group.id": f"notebook-reader-{topic}-{int(time.time())}",
        "auto.offset.reset": "earliest",
    })

    consumer.subscribe([topic])
    rows = []
    deadline = time.time() + seconds

    try:
        while len(rows) < limit and time.time() < deadline:
            msg = consumer.poll(1.0)

            if msg is None:
                continue

            if msg.error():
                print(msg.error())
                continue

            rows.append(json.loads(msg.value().decode("utf-8")))
    finally:
        consumer.close()

    return pd.DataFrame(rows)

In [28]:
borough_stats_df = consume_topic(TOPIC_BOROUGH_STATS)
borough_stats_df.head(10)

,count,window_start,window_end,trip_distance_mean,trip_distance_std,trip_distance_min,trip_distance_max,fare_amount_mean,fare_amount_std,fare_amount_min,fare_amount_max,tip_amount_mean,tip_amount_std,tip_amount_min,tip_amount_max
0,3153,1779985770000,1779985800000,2.4561,2.1447,0.0,24.90,10.2970,7.2419,0.0,118.0,1.6929,1.8074,0.0,18.00
1,72,1779985770000,1779985800000,3.0799,3.1108,0.0,16.04,17.7431,41.9765,3.0,364.5,0.9385,1.6470,0.0,6.46
2,1085,1779985770000,1779985800000,11.1494,5.7113,0.0,28.62,32.9040,14.8109,2.5,74.0,5.6903,4.9510,0.0,88.00
3,3259,1779986490000,1779986520000,2.6096,2.3409,0.0,19.17,11.1749,7.1568,2.5,67.5,1.8299,1.9961,0.0,40.30
4,84,1779986490000,1779986520000,3.8705,3.4029,0.0,20.61,19.3417,23.3710,3.0,205.0,1.7587,2.1273,0.0,6.82
5,255,1779986490000,1779986520000,10.4929,6.3953,0.0,27.90,31.8392,15.9509,2.5,74.5,3.7611,5.0948,0.0,36.00
6,10357,1779986520000,1779986550000,2.6883,2.3515,0.0,22.17,11.0279,7.0388,0.0,100.0,1.8574,2.3881,0.0,141.11
7,459,1779986520000,1779986550000,8.6506,7.0248,0.0,35.53,27.0150,18.3775,0.0,99.0,3.0466,5.6967,0.0,67.00
8,280,1779986520000,1779986550000,3.6181,2.9599,0.0,20.31,14.5532,9.2029,2.5,65.0,2.1931,7.4228,0.0,110.00
9,9915,1779986550000,1779986580000,3.3187,3.4620,0.0,28.98,12.4660,9.9366,0.0,100.0,1.7552,2.4367,0.0,60.00


In [29]:
location_stats_df = consume_topic(TOPIC_LOCATION_STATS)
location_stats_df.head(10)

,count,window_start,window_end,trip_distance_mean,trip_distance_std,trip_distance_min,trip_distance_max,fare_amount_mean,fare_amount_std,fare_amount_min,fare_amount_max,tip_amount_mean,tip_amount_std,tip_amount_min,tip_amount_max
0,470,1779985770000,1779985800000,14.6145,5.9268,0.00,28.62,40.8585,15.4124,2.5,74.0,6.1919,4.9870,0.0,20.00
1,123,1779985770000,1779985800000,2.5375,1.7859,0.00,9.76,10.5854,6.3119,3.0,52.0,1.5363,1.4862,0.0,6.56
2,384,1779985770000,1779985800000,2.7637,2.2383,0.09,17.41,11.2917,6.9043,3.0,52.0,2.2865,1.7440,0.0,12.28
3,88,1779985770000,1779985800000,2.3712,2.0514,0.30,12.32,9.6534,5.8057,3.5,37.0,1.5649,1.6376,0.0,8.20
4,137,1779985770000,1779985800000,2.5822,2.6026,0.35,19.59,10.1642,7.0120,3.5,52.5,1.8899,2.1554,0.0,18.00
5,158,1779985770000,1779985800000,2.5559,2.3035,0.00,12.51,10.7544,8.0874,3.5,64.2,1.5649,1.7231,0.0,7.95
6,121,1779985770000,1779985800000,2.9679,2.6304,0.31,16.94,11.5744,7.7839,3.5,52.0,1.9685,1.8328,0.0,7.95
7,46,1779985770000,1779985800000,2.4007,1.9175,0.33,10.50,9.4783,5.4543,3.5,34.0,1.3502,1.4010,0.0,4.96
8,141,1779985770000,1779985800000,2.0792,2.5235,0.29,24.90,8.9539,7.2432,3.5,71.5,1.1574,1.4597,0.0,5.45
9,129,1779985770000,1779985800000,2.1454,1.7671,0.20,11.42,9.2829,6.2913,0.0,52.0,1.8791,1.6577,0.0,11.00


In [30]:
python_borough_stats_df = consume_topic(TOPIC_PY_BOROUGH_STATS)
python_borough_stats_df.head(10)

,processor,group_type,group_value,count,window_start,window_end,trip_distance_mean,trip_distance_std,trip_distance_min,trip_distance_max,fare_amount_mean,fare_amount_std,fare_amount_min,fare_amount_max,tip_amount_mean,tip_amount_std,tip_amount_min,tip_amount_max
0,regular_python,borough,Manhattan,62,2019-09-01T00:00:00+00:00,2019-09-01T00:00:30+00:00,2.5287,2.2416,0.28,11.28,10.8790,6.7194,3.5,34.0,1.5602,1.5615,0.00,6.66
1,regular_python,borough,Brooklyn,2,2019-09-01T00:00:00+00:00,2019-09-01T00:00:30+00:00,5.9000,2.0800,3.82,7.98,21.2500,3.7500,17.5,25.0,4.7600,1.0000,3.76,5.76
2,regular_python,borough,Queens,8,2019-09-01T00:00:00+00:00,2019-09-01T00:00:30+00:00,12.9150,4.5312,7.68,21.23,38.8750,11.4038,21.5,52.0,5.3500,4.4621,0.00,12.25
3,regular_python,borough,Manhattan,55,2019-09-01T00:00:30+00:00,2019-09-01T00:01:00+00:00,2.8715,3.0394,0.10,19.17,11.8182,8.1572,3.5,52.0,1.8044,1.7725,0.00,6.06
4,regular_python,borough,Queens,9,2019-09-01T00:00:30+00:00,2019-09-01T00:01:00+00:00,12.4433,7.5429,2.20,25.70,35.2778,17.5707,9.5,67.5,5.1722,5.3589,0.00,13.75
5,regular_python,borough,Manhattan,55,2019-09-01T00:01:00+00:00,2019-09-01T00:01:30+00:00,2.9391,2.6459,0.50,14.98,12.3364,8.2754,3.5,51.0,1.6824,1.5275,0.00,5.05
6,regular_python,borough,Queens,10,2019-09-01T00:01:00+00:00,2019-09-01T00:01:30+00:00,10.6290,7.3452,1.40,25.32,32.7000,19.4193,6.5,67.5,1.5630,3.5928,0.00,11.78
7,regular_python,borough,Brooklyn,1,2019-09-01T00:01:00+00:00,2019-09-01T00:01:30+00:00,1.4200,0.0000,1.42,1.42,8.5000,0.0000,8.5,8.5,0.0000,0.0000,0.00,0.00
8,regular_python,borough,Queens,2,2019-09-01T00:01:30+00:00,2019-09-01T00:02:00+00:00,8.5800,0.6800,7.90,9.26,25.7500,1.7500,24.0,27.5,2.5250,2.5250,0.00,5.05
9,regular_python,borough,Manhattan,55,2019-09-01T00:01:30+00:00,2019-09-01T00:02:00+00:00,2.3540,2.1563,0.00,9.84,11.0673,7.8958,3.0,40.5,1.8482,1.8205,0.00,6.76


In [ ]:
cluster_df = consume_topic(TOPIC_CLUSTERS)
cluster_df.head(10)

,cluster_id,cluster_distance,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,PU_Borough,PU_Zone,DO_Borough,DO_Zone,features
0,0,0.0,2019-09-01T00:00:00,2019-09-01T00:17:16,229,112,Manhattan,Sutton Place/Turtle Bay North,Brooklyn,Greenpoint,"{'trip_distance': 4.19, 'fare_amount': 16.0, '..."
1,1,0.0,2019-09-01T00:00:00,2019-09-01T00:05:47,148,256,Manhattan,Lower East Side,Brooklyn,Williamsburg (South Side),"{'trip_distance': 2.0, 'fare_amount': 8.0, 'ti..."
2,2,0.0,2019-09-01T00:00:00,2019-09-01T00:26:19,256,37,Brooklyn,Williamsburg (South Side),Brooklyn,Bushwick South,"{'trip_distance': 3.82, 'fare_amount': 17.5, '..."
3,3,0.0,2019-09-01T00:00:00,2019-09-01T00:05:26,231,261,Manhattan,TriBeCa/Civic Center,Manhattan,World Trade Center,"{'trip_distance': 1.16, 'fare_amount': 6.0, 't..."
4,4,0.0,2019-09-01T00:00:00,2019-09-01T00:33:02,74,226,Manhattan,East Harlem North,Queens,Sunnyside,"{'trip_distance': 10.03, 'fare_amount': 34.0, ..."
